<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/workflow/physics_bounds_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deterministic physics and bounds checks between workflow steps

Tool calls and agent steps sometimes return numbers that cannot be physical: a probability of `1.8`, a negative mass, a speed faster than light, or a `NaN`. The next step will treat that payload as a fact unless something between the two steps rejects it.

Workflows are already event-driven. The step that receives a tool event is a natural place to run that check. This notebook builds the check with **no LLM**. Every verdict comes from a small rule table that lives in the notebook. The same payload and the same table always produce the same verdict, so a stored result can be recomputed later.

A failed check can do either of two things:

* **Stop** the run and return the verdict.
* **Emit a retry event** so the reader step may try again.

A second workflow shows the same split with a step retry policy: a transient empty read is retried, a physics violation is not.


## Install

The example uses `llama-index-core` only. The workflow runtime it re-exports supplies `@step`, events, retry policies, and `@catch_error`.


In [ ]:
%pip install llama-index-core -q

Workflow steps are async. Notebooks already have an event loop, so the cells below use `await` directly. In a script, wrap the same code in `asyncio.run()`.


In [5]:
import json
import math
from typing import Any

from llama_index.core.workflow import (
    Context,
    Event,
    StartEvent,
    StopEvent,
    Workflow,
    step,
)
from workflows import catch_error
from workflows.events import StepFailedEvent
from workflows.retry_policy import (
    retry_if_exception_type,
    retry_policy,
    stop_after_attempt,
    wait_none,
)


## A versioned rule table

The table is ordinary data. `check_bounds` does not special-case probability, mass, or speed except by reading this table.

* `probability` must sit in `[0, 1]`.
* `mass_kg` must be `>= 0`.
* `speed_m_s` must sit in `[0, c]`, with `c` in meters per second.
* Every number, whether or not its key is listed, must be finite. That rejects `NaN` and both infinities.

`version` is part of every verdict. Bump it when a bound changes so an old verdict still names the table that produced it. Keys are walked in sorted order, so two payloads with the same pairs produce the same violation list.


In [7]:
SPEED_OF_LIGHT_M_S = 299_792_458.0

PHYSICS_BOUNDS_V1 = {
    "version": "1.0.0",
    "quantities": {
        "probability": {"min": 0.0, "max": 1.0},
        "mass_kg": {"min": 0.0, "max": None},
        "speed_m_s": {"min": 0.0, "max": SPEED_OF_LIGHT_M_S},
    },
}


def _is_number(value: Any) -> bool:
    # bool is a subclass of int; a flag is not a measurement.
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def _render_number(value: int | float) -> int | float | str:
    if isinstance(value, float) and math.isnan(value):
        return "NaN"
    if isinstance(value, float) and math.isinf(value):
        return "Infinity" if value > 0 else "-Infinity"
    return value


def _path_str(path: tuple[str | int, ...]) -> str:
    text = ""
    for part in path:
        if isinstance(part, int):
            text += f"[{part}]"
        elif not text:
            text = str(part)
        else:
            text += f".{part}"
    return text


def check_bounds(payload: Any, rules: dict[str, Any] | None = None) -> dict[str, Any]:
    """Return a JSON-friendly verdict for ``payload``.

    The verdict is a pure function of ``payload`` and ``rules``. Nested dicts
    and lists are walked. A listed quantity is checked against its closed
    interval. Non-finite numbers fail first, under rule id ``finite``.
    """
    table = PHYSICS_BOUNDS_V1 if rules is None else rules
    quantities: dict[str, dict[str, float | None]] = table["quantities"]
    violations: list[dict[str, Any]] = []

    def walk(node: Any, path: tuple[str | int, ...]) -> None:
        if isinstance(node, dict):
            for key in sorted(node):
                walk(node[key], path + (key,))
            return
        if isinstance(node, list):
            for index, item in enumerate(node):
                walk(item, path + (index,))
            return
        if not _is_number(node):
            return

        location = _path_str(path)
        if not math.isfinite(float(node)):
            violations.append(
                {
                    "rule_id": "finite",
                    "path": location,
                    "value": _render_number(node),
                    "reason": "value must be a finite number",
                }
            )
            return

        leaf = path[-1] if path else ""
        spec = quantities.get(leaf) if isinstance(leaf, str) else None
        if spec is None:
            return
        low = spec["min"]
        high = spec["max"]
        if low is not None and node < low:
            reason = f"value {node} is below the minimum {low}"
        elif high is not None and node > high:
            reason = f"value {node} is above the maximum {high}"
        else:
            return
        violations.append(
            {
                "rule_id": leaf,
                "path": location,
                "value": _render_number(node),
                "reason": reason,
            }
        )

    walk(payload, ())
    return {
        "rule_table_version": table["version"],
        "accepted": not violations,
        "violations": violations,
    }


def show(verdict: dict[str, Any]) -> None:
    print(json.dumps(verdict, indent=2))


## The four failures, and one reading that passes

These calls do not start a workflow. They show that the verdict is fixed before any step runs. A tool-shaped payload uses the same keys a `ToolCall` (`tool_kwargs`) and a `ToolCallResult` (`tool_output`) would contribute. The checker only sees the dict.


In [9]:
CASES = {
    "probability above 1": {
        "tool_kwargs": {},
        "tool_output": {"probability": 1.8, "mass_kg": 1.0, "speed_m_s": 10.0},
    },
    "negative mass": {
        "tool_kwargs": {},
        "tool_output": {"probability": 0.5, "mass_kg": -2.0, "speed_m_s": 10.0},
    },
    "faster than light": {
        "tool_kwargs": {},
        "tool_output": {
            "probability": 0.5,
            "mass_kg": 1.0,
            "speed_m_s": 2 * SPEED_OF_LIGHT_M_S,
        },
    },
    "nan and infinity": {
        "tool_kwargs": {"samples": [{"mass_kg": float("nan")}]},
        "tool_output": {"speed_m_s": float("inf")},
    },
    "physically allowed": {
        "tool_kwargs": {},
        "tool_output": {"probability": 0.25, "mass_kg": 1.5, "speed_m_s": 2.9e8},
    },
}

for name, payload in CASES.items():
    print(name)
    show(check_bounds(payload))

repeated = check_bounds(CASES["probability above 1"])
print("recomputable:", repeated == check_bounds(CASES["probability above 1"]))


probability above 1
{
  "rule_table_version": "1.0.0",
  "accepted": false,
  "violations": [
    {
      "rule_id": "probability",
      "path": "tool_output.probability",
      "value": 1.8,
      "reason": "value 1.8 is above the maximum 1.0"
    }
  ]
}
negative mass
{
  "rule_table_version": "1.0.0",
  "accepted": false,
  "violations": [
    {
      "rule_id": "mass_kg",
      "path": "tool_output.mass_kg",
      "value": -2.0,
      "reason": "value -2.0 is below the minimum 0.0"
    }
  ]
}
faster than light
{
  "rule_table_version": "1.0.0",
  "accepted": false,
  "violations": [
    {
      "rule_id": "speed_m_s",
      "path": "tool_output.speed_m_s",
      "value": 599584916.0,
      "reason": "value 599584916.0 is above the maximum 299792458.0"
    }
  ]
}
nan and infinity
{
  "rule_table_version": "1.0.0",
  "accepted": false,
  "violations": [
    {
      "rule_id": "finite",
      "path": "tool_kwargs.samples[0].mass_kg",
      "value": "NaN",
      "reason": "value mus

## The workflow

Two steps, and no model:

1. `read_tool_output` stands in for a tool call. It emits the next scripted payload as a `ToolObservationEvent`. The list of payloads is fixed, so the run does not depend on a sampler.
2. `validate_bounds` is the event handler. It runs `check_bounds` and writes a `PhysicsVerdictEvent` to the stream.

If the payload is accepted, the handler returns `StopEvent`. If it is rejected and `on_violation` is `"stop"`, it also returns `StopEvent`, with `action` set to `"stop"`. If `on_violation` is `"retry"` and attempts remain, it returns `PhysicsRetryEvent`, which loops back to the reader. The last scripted payload is reused once the list runs out, and `max_attempts` still ends the run.


In [11]:
class ToolObservationEvent(Event):
    """Numeric payload from a tool or agent step."""

    tool_name: str
    payload: dict
    attempt: int


class PhysicsRetryEvent(Event):
    """Ask the reader for another payload after a rejected verdict."""

    tool_name: str
    attempt: int
    violations: list


class PhysicsVerdictEvent(Event):
    """Copy of the verdict, published on the handler stream."""

    rule_table_version: str
    accepted: bool
    violations: list
    tool_name: str
    attempt: int


class PhysicsBoundsWorkflow(Workflow):
    def __init__(
        self,
        *,
        readings: list[dict],
        on_violation: str = "stop",
        max_attempts: int = 3,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        if on_violation not in {"stop", "retry"}:
            raise ValueError("on_violation must be 'stop' or 'retry'")
        if not readings:
            raise ValueError("readings must contain at least one payload")
        self._readings = readings
        self._on_violation = on_violation
        self._max_attempts = max_attempts

    @step
    async def read_tool_output(
        self, ctx: Context, ev: StartEvent | PhysicsRetryEvent
    ) -> ToolObservationEvent:
        attempt = 0 if isinstance(ev, StartEvent) else ev.attempt
        index = min(attempt, len(self._readings) - 1)
        observed = ToolObservationEvent(
            tool_name=ev.tool_name,
            payload=self._readings[index],
            attempt=attempt,
        )
        ctx.write_event_to_stream(observed)
        return observed

    @step
    async def validate_bounds(
        self, ctx: Context, ev: ToolObservationEvent
    ) -> PhysicsRetryEvent | StopEvent:
        verdict = check_bounds(ev.payload)
        ctx.write_event_to_stream(
            PhysicsVerdictEvent(
                rule_table_version=verdict["rule_table_version"],
                accepted=verdict["accepted"],
                violations=verdict["violations"],
                tool_name=ev.tool_name,
                attempt=ev.attempt,
            )
        )
        should_stop = (
            verdict["accepted"]
            or self._on_violation == "stop"
            or ev.attempt + 1 >= self._max_attempts
        )
        if should_stop:
            action = "accept" if verdict["accepted"] else "stop"
            return StopEvent(
                result={
                    **verdict,
                    "tool_name": ev.tool_name,
                    "attempt": ev.attempt,
                    "action": action,
                }
            )
        return PhysicsRetryEvent(
            tool_name=ev.tool_name,
            attempt=ev.attempt + 1,
            violations=verdict["violations"],
        )


## Stop on the first physics violation

Each run uses a fresh workflow and one bad reading. The result is the verdict plus which tool produced it and which attempt stopped the run.


In [13]:
async def run_bounds(
    readings: list[dict],
    *,
    tool_name: str,
    on_violation: str = "stop",
    max_attempts: int = 3,
) -> dict:
    workflow = PhysicsBoundsWorkflow(
        readings=readings,
        on_violation=on_violation,
        max_attempts=max_attempts,
        timeout=30,
    )
    handler = workflow.run(tool_name=tool_name)
    async for event in handler.stream_events():
        if isinstance(event, ToolObservationEvent):
            print(f"stream observe attempt={event.attempt} tool={event.tool_name}")
        elif isinstance(event, PhysicsVerdictEvent):
            rules = [item["rule_id"] for item in event.violations]
            print(
                f"stream verdict attempt={event.attempt} "
                f"accepted={event.accepted} rules={rules}"
            )
    return await handler


In [14]:
stop_cases = {
    "probability": CASES["probability above 1"],
    "mass": CASES["negative mass"],
    "speed": CASES["faster than light"],
    "non_finite": CASES["nan and infinity"],
    "valid": CASES["physically allowed"],
}
for name, payload in stop_cases.items():
    print("==", name)
    show(
        await run_bounds(
            [payload],
            tool_name=f"{name}_sensor",
            on_violation="stop",
        )
    )


== probability
stream observe attempt=0 tool=probability_sensor
stream verdict attempt=0 accepted=False rules=['probability']
{
  "rule_table_version": "1.0.0",
  "accepted": false,
  "violations": [
    {
      "rule_id": "probability",
      "path": "tool_output.probability",
      "value": 1.8,
      "reason": "value 1.8 is above the maximum 1.0"
    }
  ],
  "tool_name": "probability_sensor",
  "attempt": 0,
  "action": "stop"
}
== mass
stream observe attempt=0 tool=mass_sensor
stream verdict attempt=0 accepted=False rules=['mass_kg']
{
  "rule_table_version": "1.0.0",
  "accepted": false,
  "violations": [
    {
      "rule_id": "mass_kg",
      "path": "tool_output.mass_kg",
      "value": -2.0,
      "reason": "value -2.0 is below the minimum 0.0"
    }
  ],
  "tool_name": "mass_sensor",
  "attempt": 0,
  "action": "stop"
}
== speed
stream observe attempt=0 tool=speed_sensor
stream verdict attempt=0 accepted=False rules=['speed_m_s']
{
  "rule_table_version": "1.0.0",
  "accepte

## Emit a retry event instead of stopping

The first reading is faster than light. The second is allowed. With `on_violation="retry"` the handler sends `PhysicsRetryEvent`, the reader emits the next scripted payload, and the run stops only when the verdict is accepted.

A second run never gets a legal reading. `max_attempts=2` stops it with the last rejection rather than looping forever.


In [16]:
corrected = {
    "tool_kwargs": {},
    "tool_output": {"probability": 0.2, "mass_kg": 1.0, "speed_m_s": 10.0},
}
print("== retry then accept")
show(
    await run_bounds(
        [CASES["faster than light"], corrected],
        tool_name="speed_sensor",
        on_violation="retry",
        max_attempts=3,
    )
)

print("== retry exhausted")
show(
    await run_bounds(
        [CASES["probability above 1"], CASES["probability above 1"]],
        tool_name="probability_sensor",
        on_violation="retry",
        max_attempts=2,
    )
)


== retry then accept
stream observe attempt=0 tool=speed_sensor
stream verdict attempt=0 accepted=False rules=['speed_m_s']
stream observe attempt=1 tool=speed_sensor
stream verdict attempt=1 accepted=True rules=[]
{
  "rule_table_version": "1.0.0",
  "accepted": true,
  "violations": [],
  "tool_name": "speed_sensor",
  "attempt": 1,
  "action": "accept"
}
== retry exhausted
stream observe attempt=0 tool=probability_sensor
stream verdict attempt=0 accepted=False rules=['probability']
stream observe attempt=1 tool=probability_sensor
stream verdict attempt=1 accepted=False rules=['probability']
{
  "rule_table_version": "1.0.0",
  "accepted": false,
  "violations": [
    {
      "rule_id": "probability",
      "path": "tool_output.probability",
      "value": 1.8,
      "reason": "value 1.8 is above the maximum 1.0"
    }
  ],
  "tool_name": "probability_sensor",
  "attempt": 1,
  "action": "stop"
}


## Compose the same check with a retry policy

`retry_policy` retries a step when it raises. That is a different knob from the retry *event* above.

* `TransientSensorError` means the tool returned nothing. The policy retries it, with no delay, up to 3 attempts.
* `PhysicsViolationError` means the tool returned a number the table rejects. The policy does not retry that exception. Retrying would call the same tool again on data that is already known to be impossible.

`@catch_error` is the handler for a step that has finished failing. It turns the exception into a `StopEvent` so the caller still receives a verdict instead of an uncaught error. Build a new workflow instance per run: the script cursor lives on `self`.


In [18]:
class TransientSensorError(Exception):
    """The tool returned no payload. Calling it again may succeed."""


class PhysicsViolationError(Exception):
    """The payload is complete and the rule table rejected it."""

    def __init__(self, verdict: dict[str, Any]) -> None:
        self.verdict = verdict
        rule_ids = ", ".join(item["rule_id"] for item in verdict["violations"])
        super().__init__(
            f"physics violation ({verdict['rule_table_version']}): {rule_ids}"
        )


# Retry only empty reads. A physics violation falls through on the first attempt.
TRANSIENT_READ_POLICY = retry_policy(
    retry=retry_if_exception_type(TransientSensorError),
    wait=wait_none(),
    stop=stop_after_attempt(3),
)


class RetryPolicyPhysicsWorkflow(Workflow):
    def __init__(self, script: list[dict | None], **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self._script = list(script)
        self._index = 0

    @step(retry_policy=TRANSIENT_READ_POLICY)
    async def read_and_check(self, ev: StartEvent) -> StopEvent:
        item = self._script[self._index]
        self._index += 1
        if item is None:
            raise TransientSensorError("tool returned no payload")
        verdict = check_bounds(item)
        if not verdict["accepted"]:
            raise PhysicsViolationError(verdict)
        return StopEvent(
            result={**verdict, "action": "accept", "attempt": self._index - 1}
        )

    @catch_error(for_steps=["read_and_check"])
    async def on_check_failed(self, ev: StepFailedEvent) -> StopEvent:
        error = ev.exception
        verdict = getattr(error, "verdict", None)
        return StopEvent(
            result={
                "accepted": False,
                "action": "stop",
                "attempts": ev.attempts,
                "error_type": type(error).__name__,
                "rule_table_version": None
                if verdict is None
                else verdict["rule_table_version"],
                "verdict": verdict,
            }
        )


async def run_retry_policy(script: list[dict | None]) -> dict:
    workflow = RetryPolicyPhysicsWorkflow(script, timeout=30)
    return await workflow.run(tool_name="sensor")


print("== transient reads, then a legal payload")
show(
    await run_retry_policy(
        [None, None, CASES["physically allowed"]],
    )
)

print("== probability 1.8 is not retried")
show(await run_retry_policy([CASES["probability above 1"]]))

print("== empty reads exhausted")
show(await run_retry_policy([None, None, None]))


== transient reads, then a legal payload
{
  "rule_table_version": "1.0.0",
  "accepted": true,
  "violations": [],
  "action": "accept",
  "attempt": 2
}
== probability 1.8 is not retried
{
  "accepted": false,
  "action": "stop",
  "attempts": 1,
  "error_type": "PhysicsViolationError",
  "rule_table_version": "1.0.0",
  "verdict": {
    "rule_table_version": "1.0.0",
    "accepted": false,
    "violations": [
      {
        "rule_id": "probability",
        "path": "tool_output.probability",
        "value": 1.8,
        "reason": "value 1.8 is above the maximum 1.0"
      }
    ]
  }
}
== empty reads exhausted
{
  "accepted": false,
  "action": "stop",
  "attempts": 3,
  "error_type": "TransientSensorError",
  "rule_table_version": null,
  "verdict": null
}


## Bump the table version when a bound changes

A probability of `0.7` passes version `1.0.0`. A copied table that caps probability at `0.5` rejects it, and the verdict names `1.1.0`. Recomputing with the original table still accepts it. The handler code does not change.


In [20]:
reading = {"probability": 0.7, "mass_kg": 1.0, "speed_m_s": 10.0}
stricter = {
    "version": "1.1.0",
    "quantities": {
        **PHYSICS_BOUNDS_V1["quantities"],
        "probability": {"min": 0.0, "max": 0.5},
    },
}
print("table 1.0.0")
show(check_bounds(reading))
print("table 1.1.0")
show(check_bounds(reading, stricter))


table 1.0.0
{
  "rule_table_version": "1.0.0",
  "accepted": true,
  "violations": []
}
table 1.1.0
{
  "rule_table_version": "1.1.0",
  "accepted": false,
  "violations": [
    {
      "rule_id": "probability",
      "path": "probability",
      "value": 0.7,
      "reason": "value 0.7 is above the maximum 0.5"
    }
  ]
}


## What to keep

Put the check in the step that receives the tool or agent event, and return the verdict from `StopEvent` when the run must end. Emit a separate event when another step is allowed to try again. Use a retry policy only for failures that might succeed on another call, and keep physics violations out of that policy.

The rule table in this notebook is the whole policy. Change a number, bump `version`, and `check_bounds` will reproduce every verdict from the payload plus that table. No model call is required.
